In [ ]:
from pathlib import Path
import sys

project_root = Path.cwd().resolve()
if not (project_root / "ProofOfConcept").exists():
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import pandas as pd
from ProofOfConcept.download_images import download_structure_samples

metadata = pd.read_csv(project_root / "ProofOfConcept" / "metadata.csv")
structure_names = [
    name.strip() for name in (project_root / "ProofOfConcept" / "structure_names.txt").read_text(encoding="utf-8").splitlines() if name.strip()
]
download_structure_samples(
    structure_names,
    num_stacks=3,
    output_dir=project_root / "ProofOfConcept" / "structure_downloads",
    metadata=metadata,
)
print(f"Downloaded 3 TIFF pairs for {len(structure_names)} structures into {project_root / 'ProofOfConcept' / 'structure_downloads'}")


C:\Users\Adithi\AppData\Local\Temp\ipykernel_19608\3726913504.py:13: DtypeWarning: Columns (0: meta_colony_centroid) have mixed types. Specify dtype option on import or set low_memory=False.
  metadata = pd.read_csv(project_root / "ProofOfConcept" / "metadata.csv")
Loading manifest: 100%|██████████| 484465/484465 [00:07<00:00, 67.6k/s]


Ensuring 3 stack(s) are available for AAVS1...
Ensuring 3 stack(s) are available for ACTB...
Ensuring 3 stack(s) are available for ACTN1...
Ensuring 3 stack(s) are available for ATP2A2...
Ensuring 3 stack(s) are available for CETN2...
Ensuring 3 stack(s) are available for CTNNB1...
Ensuring 3 stack(s) are available for DSP...
Ensuring 3 stack(s) are available for FBL...
Ensuring 3 stack(s) are available for GJA1...
Ensuring 3 stack(s) are available for HIST1H2BJ...
Ensuring 3 stack(s) are available for LAMP1...
Ensuring 3 stack(s) are available for LMNB1...
Ensuring 3 stack(s) are available for MYH10...
Ensuring 3 stack(s) are available for NPM1...
Ensuring 3 stack(s) are available for NUP153...
Ensuring 3 stack(s) are available for PXN...
Ensuring 3 stack(s) are available for RAB5A...
Ensuring 3 stack(s) are available for SEC61B...
Ensuring 3 stack(s) are available for SLC25A17...
Ensuring 3 stack(s) are available for SMC1A...
Ensuring 3 stack(s) are available for SON...
Ensuring 3 st

In [30]:
from pathlib import Path
import re
from PIL import Image


def compose_structure_gallery(structure_name: str, output_root: Path) -> Path:
    structure_dir = output_root / structure_name
    if not structure_dir.exists():
        raise FileNotFoundError(f"Missing output directory for {structure_name}: {structure_dir}")

    raw_files = sorted(structure_dir.glob(f"{structure_name}_raw_*_z*.png"))
    mask_files = sorted(structure_dir.glob(f"{structure_name}_mask_*_z*.png"))
    if not raw_files or not mask_files:
        raise FileNotFoundError(
            f"No raw/mask PNGs found in {structure_dir} for {structure_name}. "
            "Run the per-structure preview cell first."
        )

    matched_pairs = []
    for raw_path in raw_files:
        match = re.search(r"z(-?\d+)\.png$", raw_path.name)
        if match is None:
            continue
        z_value = int(match.group(1))
        mask_candidates = [
            p for p in mask_files if re.search(rf"z{z_value}\.png$", p.name)
        ]
        if not mask_candidates:
            continue
        matched_pairs.append((raw_path, sorted(mask_candidates)[0], z_value))

    if not matched_pairs:
        raise FileNotFoundError(
            f"No matching raw/mask pairs found for {structure_name} in {structure_dir}."
        )

    matched_pairs.sort(key=lambda item: item[2])
    first_raw = Image.open(matched_pairs[0][0]).convert("RGB")
    width, height = first_raw.size

    montage = Image.new("RGB", (width * len(matched_pairs), height * 2))
    for column_index, (raw_path, mask_path, _) in enumerate(matched_pairs):
        raw_image = Image.open(raw_path).convert("RGB")
        mask_image = Image.open(mask_path).convert("RGB")
        montage.paste(raw_image, (column_index * width, 0))
        montage.paste(mask_image, (column_index * width, height))

    output_path = structure_dir / f"{structure_name}_raw_mask_composite.png"
    montage.save(output_path)
    print(f"Saved composite for {structure_name}: {output_path}")
    return output_path


for structure_name in structure_names:
    try:
        compose_structure_gallery(structure_name, project_root / "visualizations")
    except FileNotFoundError as exc:
        print(exc)


Saved composite for AAVS1: C:\Users\Adithi\Code\PyCode\Self\Prof. Ashesh Project\visualizations\AAVS1\AAVS1_raw_mask_composite.png
Saved composite for ACTB: C:\Users\Adithi\Code\PyCode\Self\Prof. Ashesh Project\visualizations\ACTB\ACTB_raw_mask_composite.png
Saved composite for ACTN1: C:\Users\Adithi\Code\PyCode\Self\Prof. Ashesh Project\visualizations\ACTN1\ACTN1_raw_mask_composite.png
Saved composite for ATP2A2: C:\Users\Adithi\Code\PyCode\Self\Prof. Ashesh Project\visualizations\ATP2A2\ATP2A2_raw_mask_composite.png
Saved composite for CETN2: C:\Users\Adithi\Code\PyCode\Self\Prof. Ashesh Project\visualizations\CETN2\CETN2_raw_mask_composite.png
Saved composite for CTNNB1: C:\Users\Adithi\Code\PyCode\Self\Prof. Ashesh Project\visualizations\CTNNB1\CTNNB1_raw_mask_composite.png
Saved composite for DSP: C:\Users\Adithi\Code\PyCode\Self\Prof. Ashesh Project\visualizations\DSP\DSP_raw_mask_composite.png
Saved composite for FBL: C:\Users\Adithi\Code\PyCode\Self\Prof. Ashesh Project\visuali

## AAVS1


In [ ]:
from pathlib import Path
import random
import sys

project_root = Path.cwd().resolve()
if not (project_root / "ProofOfConcept").exists():
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import pandas as pd
from ProofOfConcept.visualize_structures import visualize_structure_samples

structure_name = "AAVS1"
metadata = pd.read_csv(project_root / "ProofOfConcept" / "metadata.csv")
random_seed = random.randint(1, 10**9)

selected = visualize_structure_samples(
    structure_name,
    metadata=metadata,
    input_dir=project_root / "ProofOfConcept" / "structure_downloads",
    output_dir=project_root / "visualizations",
    num_files=3,
    random_seed=random_seed,
    z_offsets=[-1, 0, 1],
)

print(f"{structure_name}: random_seed={random_seed}; saved {len(selected)} raw/mask pairs")
for raw_path, mask_path, z_offset in selected:
    print(f"{raw_path.name} | {mask_path.name} | z_offset={z_offset}")


C:\Users\Adithi\AppData\Local\Temp\ipykernel_19608\3322180349.py:15: DtypeWarning: Columns (0: meta_colony_centroid) have mixed types. Specify dtype option on import or set low_memory=False.
  metadata = pd.read_csv(project_root / "ProofOfConcept" / "metadata.csv")


AAVS1: random_seed=965536462; saved 3 raw/mask pairs
AAVS1_raw_1_z-1.png | AAVS1_mask_1_z-1.png | z_offset=-1
AAVS1_raw_2_z0.png | AAVS1_mask_2_z0.png | z_offset=0
AAVS1_raw_3_z1.png | AAVS1_mask_3_z1.png | z_offset=1


## ACTB


In [6]:
from pathlib import Path
import random
import sys

project_root = Path.cwd().resolve()
if not (project_root / "ProofOfConcept").exists():
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import pandas as pd
from ProofOfConcept.visualize_structures import visualize_structure_samples

structure_name = "ACTB"
metadata = pd.read_csv(project_root / "ProofOfConcept" / "metadata.csv")
random_seed = random.randint(1, 10**9)

selected = visualize_structure_samples(
    structure_name,
    metadata=metadata,
    input_dir=project_root / "ProofOfConcept" / "structure_downloads",
    output_dir=project_root / "visualizations",
    num_files=3,
    random_seed=random_seed,
    z_offsets=[-1, 0, 1],
)

print(f"{structure_name}: random_seed={random_seed}; saved {len(selected)} raw/mask pairs")
for raw_path, mask_path, z_offset in selected:
    print(f"{raw_path.name} | {mask_path.name} | z_offset={z_offset}")


C:\Users\Adithi\AppData\Local\Temp\ipykernel_19608\460016146.py:15: DtypeWarning: Columns (0: meta_colony_centroid) have mixed types. Specify dtype option on import or set low_memory=False.
  metadata = pd.read_csv(project_root / "ProofOfConcept" / "metadata.csv")
<tifffile.TiffFrame 174 @14965256> is missing required tags
<tifffile.TiffFile 'f5365e1a1248a292…10c_raw.ome.tif'> OME series contains index out of range, raised IndexError('index 175 out of range(175)')
<tifffile.TiffFile 'f5365e1a1248a292…10c_raw.ome.tif'> OME series contains index out of range, raised IndexError('index 176 out of range(175)')
<tifffile.TiffFile 'f5365e1a1248a292…10c_raw.ome.tif'> OME series contains index out of range, raised IndexError('index 177 out of range(175)')
<tifffile.TiffFile 'f5365e1a1248a292…10c_raw.ome.tif'> OME series contains index out of range, raised IndexError('index 178 out of range(175)')
<tifffile.TiffFile 'f5365e1a1248a292…10c_raw.ome.tif'> OME series contains index out of range, rais

RuntimeError: incompatible keyframe

## ACTN1


In [7]:
from pathlib import Path
import random
import sys

project_root = Path.cwd().resolve()
if not (project_root / "ProofOfConcept").exists():
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import pandas as pd
from ProofOfConcept.visualize_structures import visualize_structure_samples

structure_name = "ACTN1"
metadata = pd.read_csv(project_root / "ProofOfConcept" / "metadata.csv")
random_seed = random.randint(1, 10**9)

selected = visualize_structure_samples(
    structure_name,
    metadata=metadata,
    input_dir=project_root / "ProofOfConcept" / "structure_downloads",
    output_dir=project_root / "visualizations",
    num_files=3,
    random_seed=random_seed,
    z_offsets=[-1, 0, 1],
)

print(f"{structure_name}: random_seed={random_seed}; saved {len(selected)} raw/mask pairs")
for raw_path, mask_path, z_offset in selected:
    print(f"{raw_path.name} | {mask_path.name} | z_offset={z_offset}")


C:\Users\Adithi\AppData\Local\Temp\ipykernel_19608\4218490549.py:15: DtypeWarning: Columns (0: meta_colony_centroid) have mixed types. Specify dtype option on import or set low_memory=False.
  metadata = pd.read_csv(project_root / "ProofOfConcept" / "metadata.csv")


ACTN1: random_seed=655126145; saved 3 raw/mask pairs
ACTN1_raw_1_z-1.png | ACTN1_mask_1_z-1.png | z_offset=-1
ACTN1_raw_2_z0.png | ACTN1_mask_2_z0.png | z_offset=0
ACTN1_raw_3_z1.png | ACTN1_mask_3_z1.png | z_offset=1


## ATP2A2


In [8]:
from pathlib import Path
import random
import sys

project_root = Path.cwd().resolve()
if not (project_root / "ProofOfConcept").exists():
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import pandas as pd
from ProofOfConcept.visualize_structures import visualize_structure_samples

structure_name = "ATP2A2"
metadata = pd.read_csv(project_root / "ProofOfConcept" / "metadata.csv")
random_seed = random.randint(1, 10**9)

selected = visualize_structure_samples(
    structure_name,
    metadata=metadata,
    input_dir=project_root / "ProofOfConcept" / "structure_downloads",
    output_dir=project_root / "visualizations",
    num_files=3,
    random_seed=random_seed,
    z_offsets=[-1, 0, 1],
)

print(f"{structure_name}: random_seed={random_seed}; saved {len(selected)} raw/mask pairs")
for raw_path, mask_path, z_offset in selected:
    print(f"{raw_path.name} | {mask_path.name} | z_offset={z_offset}")


C:\Users\Adithi\AppData\Local\Temp\ipykernel_19608\133726857.py:15: DtypeWarning: Columns (0: meta_colony_centroid) have mixed types. Specify dtype option on import or set low_memory=False.
  metadata = pd.read_csv(project_root / "ProofOfConcept" / "metadata.csv")


ATP2A2: random_seed=377151515; saved 3 raw/mask pairs
ATP2A2_raw_1_z-1.png | ATP2A2_mask_1_z-1.png | z_offset=-1
ATP2A2_raw_2_z0.png | ATP2A2_mask_2_z0.png | z_offset=0
ATP2A2_raw_3_z1.png | ATP2A2_mask_3_z1.png | z_offset=1


## CETN2


In [9]:
from pathlib import Path
import random
import sys

project_root = Path.cwd().resolve()
if not (project_root / "ProofOfConcept").exists():
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import pandas as pd
from ProofOfConcept.visualize_structures import visualize_structure_samples

structure_name = "CETN2"
metadata = pd.read_csv(project_root / "ProofOfConcept" / "metadata.csv")
random_seed = random.randint(1, 10**9)

selected = visualize_structure_samples(
    structure_name,
    metadata=metadata,
    input_dir=project_root / "ProofOfConcept" / "structure_downloads",
    output_dir=project_root / "visualizations",
    num_files=3,
    random_seed=random_seed,
    z_offsets=[-1, 0, 1],
)

print(f"{structure_name}: random_seed={random_seed}; saved {len(selected)} raw/mask pairs")
for raw_path, mask_path, z_offset in selected:
    print(f"{raw_path.name} | {mask_path.name} | z_offset={z_offset}")


C:\Users\Adithi\AppData\Local\Temp\ipykernel_19608\3839935289.py:15: DtypeWarning: Columns (0: meta_colony_centroid) have mixed types. Specify dtype option on import or set low_memory=False.
  metadata = pd.read_csv(project_root / "ProofOfConcept" / "metadata.csv")


CETN2: random_seed=611497954; saved 3 raw/mask pairs
CETN2_raw_1_z-1.png | CETN2_mask_1_z-1.png | z_offset=-1
CETN2_raw_2_z0.png | CETN2_mask_2_z0.png | z_offset=0
CETN2_raw_3_z1.png | CETN2_mask_3_z1.png | z_offset=1


## CTNNB1


In [10]:
from pathlib import Path
import random
import sys

project_root = Path.cwd().resolve()
if not (project_root / "ProofOfConcept").exists():
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import pandas as pd
from ProofOfConcept.visualize_structures import visualize_structure_samples

structure_name = "CTNNB1"
metadata = pd.read_csv(project_root / "ProofOfConcept" / "metadata.csv")
random_seed = random.randint(1, 10**9)

selected = visualize_structure_samples(
    structure_name,
    metadata=metadata,
    input_dir=project_root / "ProofOfConcept" / "structure_downloads",
    output_dir=project_root / "visualizations",
    num_files=3,
    random_seed=random_seed,
    z_offsets=[-1, 0, 1],
)

print(f"{structure_name}: random_seed={random_seed}; saved {len(selected)} raw/mask pairs")
for raw_path, mask_path, z_offset in selected:
    print(f"{raw_path.name} | {mask_path.name} | z_offset={z_offset}")


C:\Users\Adithi\AppData\Local\Temp\ipykernel_19608\1315639635.py:15: DtypeWarning: Columns (0: meta_colony_centroid) have mixed types. Specify dtype option on import or set low_memory=False.
  metadata = pd.read_csv(project_root / "ProofOfConcept" / "metadata.csv")


CTNNB1: random_seed=523337449; saved 3 raw/mask pairs
CTNNB1_raw_1_z-1.png | CTNNB1_mask_1_z-1.png | z_offset=-1
CTNNB1_raw_2_z0.png | CTNNB1_mask_2_z0.png | z_offset=0
CTNNB1_raw_3_z1.png | CTNNB1_mask_3_z1.png | z_offset=1


## DSP


In [11]:
from pathlib import Path
import random
import sys

project_root = Path.cwd().resolve()
if not (project_root / "ProofOfConcept").exists():
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import pandas as pd
from ProofOfConcept.visualize_structures import visualize_structure_samples

structure_name = "DSP"
metadata = pd.read_csv(project_root / "ProofOfConcept" / "metadata.csv")
random_seed = random.randint(1, 10**9)

selected = visualize_structure_samples(
    structure_name,
    metadata=metadata,
    input_dir=project_root / "ProofOfConcept" / "structure_downloads",
    output_dir=project_root / "visualizations",
    num_files=3,
    random_seed=random_seed,
    z_offsets=[-1, 0, 1],
)

print(f"{structure_name}: random_seed={random_seed}; saved {len(selected)} raw/mask pairs")
for raw_path, mask_path, z_offset in selected:
    print(f"{raw_path.name} | {mask_path.name} | z_offset={z_offset}")


C:\Users\Adithi\AppData\Local\Temp\ipykernel_19608\875617496.py:15: DtypeWarning: Columns (0: meta_colony_centroid) have mixed types. Specify dtype option on import or set low_memory=False.
  metadata = pd.read_csv(project_root / "ProofOfConcept" / "metadata.csv")


DSP: random_seed=446201289; saved 3 raw/mask pairs
DSP_raw_1_z-1.png | DSP_mask_1_z-1.png | z_offset=-1
DSP_raw_2_z0.png | DSP_mask_2_z0.png | z_offset=0
DSP_raw_3_z1.png | DSP_mask_3_z1.png | z_offset=1


## FBL


In [12]:
from pathlib import Path
import random
import sys

project_root = Path.cwd().resolve()
if not (project_root / "ProofOfConcept").exists():
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import pandas as pd
from ProofOfConcept.visualize_structures import visualize_structure_samples

structure_name = "FBL"
metadata = pd.read_csv(project_root / "ProofOfConcept" / "metadata.csv")
random_seed = random.randint(1, 10**9)

selected = visualize_structure_samples(
    structure_name,
    metadata=metadata,
    input_dir=project_root / "ProofOfConcept" / "structure_downloads",
    output_dir=project_root / "visualizations",
    num_files=3,
    random_seed=random_seed,
    z_offsets=[-1, 0, 1],
)

print(f"{structure_name}: random_seed={random_seed}; saved {len(selected)} raw/mask pairs")
for raw_path, mask_path, z_offset in selected:
    print(f"{raw_path.name} | {mask_path.name} | z_offset={z_offset}")


C:\Users\Adithi\AppData\Local\Temp\ipykernel_19608\947264161.py:15: DtypeWarning: Columns (0: meta_colony_centroid) have mixed types. Specify dtype option on import or set low_memory=False.
  metadata = pd.read_csv(project_root / "ProofOfConcept" / "metadata.csv")


FBL: random_seed=976703890; saved 3 raw/mask pairs
FBL_raw_1_z-1.png | FBL_mask_1_z-1.png | z_offset=-1
FBL_raw_2_z0.png | FBL_mask_2_z0.png | z_offset=0
FBL_raw_3_z1.png | FBL_mask_3_z1.png | z_offset=1


## GJA1


In [13]:
from pathlib import Path
import random
import sys

project_root = Path.cwd().resolve()
if not (project_root / "ProofOfConcept").exists():
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import pandas as pd
from ProofOfConcept.visualize_structures import visualize_structure_samples

structure_name = "GJA1"
metadata = pd.read_csv(project_root / "ProofOfConcept" / "metadata.csv")
random_seed = random.randint(1, 10**9)

selected = visualize_structure_samples(
    structure_name,
    metadata=metadata,
    input_dir=project_root / "ProofOfConcept" / "structure_downloads",
    output_dir=project_root / "visualizations",
    num_files=3,
    random_seed=random_seed,
    z_offsets=[-1, 0, 1],
)

print(f"{structure_name}: random_seed={random_seed}; saved {len(selected)} raw/mask pairs")
for raw_path, mask_path, z_offset in selected:
    print(f"{raw_path.name} | {mask_path.name} | z_offset={z_offset}")


C:\Users\Adithi\AppData\Local\Temp\ipykernel_19608\4094823502.py:15: DtypeWarning: Columns (0: meta_colony_centroid) have mixed types. Specify dtype option on import or set low_memory=False.
  metadata = pd.read_csv(project_root / "ProofOfConcept" / "metadata.csv")


GJA1: random_seed=533759577; saved 3 raw/mask pairs
GJA1_raw_1_z-1.png | GJA1_mask_1_z-1.png | z_offset=-1
GJA1_raw_2_z0.png | GJA1_mask_2_z0.png | z_offset=0
GJA1_raw_3_z1.png | GJA1_mask_3_z1.png | z_offset=1


## HIST1H2BJ


In [14]:
from pathlib import Path
import random
import sys

project_root = Path.cwd().resolve()
if not (project_root / "ProofOfConcept").exists():
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import pandas as pd
from ProofOfConcept.visualize_structures import visualize_structure_samples

structure_name = "HIST1H2BJ"
metadata = pd.read_csv(project_root / "ProofOfConcept" / "metadata.csv")
random_seed = random.randint(1, 10**9)

selected = visualize_structure_samples(
    structure_name,
    metadata=metadata,
    input_dir=project_root / "ProofOfConcept" / "structure_downloads",
    output_dir=project_root / "visualizations",
    num_files=3,
    random_seed=random_seed,
    z_offsets=[-1, 0, 1],
)

print(f"{structure_name}: random_seed={random_seed}; saved {len(selected)} raw/mask pairs")
for raw_path, mask_path, z_offset in selected:
    print(f"{raw_path.name} | {mask_path.name} | z_offset={z_offset}")


C:\Users\Adithi\AppData\Local\Temp\ipykernel_19608\1726536027.py:15: DtypeWarning: Columns (0: meta_colony_centroid) have mixed types. Specify dtype option on import or set low_memory=False.
  metadata = pd.read_csv(project_root / "ProofOfConcept" / "metadata.csv")


HIST1H2BJ: random_seed=328513409; saved 3 raw/mask pairs
HIST1H2BJ_raw_1_z-1.png | HIST1H2BJ_mask_1_z-1.png | z_offset=-1
HIST1H2BJ_raw_2_z0.png | HIST1H2BJ_mask_2_z0.png | z_offset=0
HIST1H2BJ_raw_3_z1.png | HIST1H2BJ_mask_3_z1.png | z_offset=1


## LAMP1


In [15]:
from pathlib import Path
import random
import sys

project_root = Path.cwd().resolve()
if not (project_root / "ProofOfConcept").exists():
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import pandas as pd
from ProofOfConcept.visualize_structures import visualize_structure_samples

structure_name = "LAMP1"
metadata = pd.read_csv(project_root / "ProofOfConcept" / "metadata.csv")
random_seed = random.randint(1, 10**9)

selected = visualize_structure_samples(
    structure_name,
    metadata=metadata,
    input_dir=project_root / "ProofOfConcept" / "structure_downloads",
    output_dir=project_root / "visualizations",
    num_files=3,
    random_seed=random_seed,
    z_offsets=[-1, 0, 1],
)

print(f"{structure_name}: random_seed={random_seed}; saved {len(selected)} raw/mask pairs")
for raw_path, mask_path, z_offset in selected:
    print(f"{raw_path.name} | {mask_path.name} | z_offset={z_offset}")


C:\Users\Adithi\AppData\Local\Temp\ipykernel_19608\1382634215.py:15: DtypeWarning: Columns (0: meta_colony_centroid) have mixed types. Specify dtype option on import or set low_memory=False.
  metadata = pd.read_csv(project_root / "ProofOfConcept" / "metadata.csv")


LAMP1: random_seed=929665849; saved 3 raw/mask pairs
LAMP1_raw_1_z-1.png | LAMP1_mask_1_z-1.png | z_offset=-1
LAMP1_raw_2_z0.png | LAMP1_mask_2_z0.png | z_offset=0
LAMP1_raw_3_z1.png | LAMP1_mask_3_z1.png | z_offset=1


## LMNB1


In [16]:
from pathlib import Path
import random
import sys

project_root = Path.cwd().resolve()
if not (project_root / "ProofOfConcept").exists():
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import pandas as pd
from ProofOfConcept.visualize_structures import visualize_structure_samples

structure_name = "LMNB1"
metadata = pd.read_csv(project_root / "ProofOfConcept" / "metadata.csv")
random_seed = random.randint(1, 10**9)

selected = visualize_structure_samples(
    structure_name,
    metadata=metadata,
    input_dir=project_root / "ProofOfConcept" / "structure_downloads",
    output_dir=project_root / "visualizations",
    num_files=3,
    random_seed=random_seed,
    z_offsets=[-1, 0, 1],
)

print(f"{structure_name}: random_seed={random_seed}; saved {len(selected)} raw/mask pairs")
for raw_path, mask_path, z_offset in selected:
    print(f"{raw_path.name} | {mask_path.name} | z_offset={z_offset}")


C:\Users\Adithi\AppData\Local\Temp\ipykernel_19608\4132198706.py:15: DtypeWarning: Columns (0: meta_colony_centroid) have mixed types. Specify dtype option on import or set low_memory=False.
  metadata = pd.read_csv(project_root / "ProofOfConcept" / "metadata.csv")


LMNB1: random_seed=702650978; saved 3 raw/mask pairs
LMNB1_raw_1_z-1.png | LMNB1_mask_1_z-1.png | z_offset=-1
LMNB1_raw_2_z0.png | LMNB1_mask_2_z0.png | z_offset=0
LMNB1_raw_3_z1.png | LMNB1_mask_3_z1.png | z_offset=1


## MYH10


In [17]:
from pathlib import Path
import random
import sys

project_root = Path.cwd().resolve()
if not (project_root / "ProofOfConcept").exists():
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import pandas as pd
from ProofOfConcept.visualize_structures import visualize_structure_samples

structure_name = "MYH10"
metadata = pd.read_csv(project_root / "ProofOfConcept" / "metadata.csv")
random_seed = random.randint(1, 10**9)

selected = visualize_structure_samples(
    structure_name,
    metadata=metadata,
    input_dir=project_root / "ProofOfConcept" / "structure_downloads",
    output_dir=project_root / "visualizations",
    num_files=3,
    random_seed=random_seed,
    z_offsets=[-1, 0, 1],
)

print(f"{structure_name}: random_seed={random_seed}; saved {len(selected)} raw/mask pairs")
for raw_path, mask_path, z_offset in selected:
    print(f"{raw_path.name} | {mask_path.name} | z_offset={z_offset}")


C:\Users\Adithi\AppData\Local\Temp\ipykernel_19608\3587159412.py:15: DtypeWarning: Columns (0: meta_colony_centroid) have mixed types. Specify dtype option on import or set low_memory=False.
  metadata = pd.read_csv(project_root / "ProofOfConcept" / "metadata.csv")


MYH10: random_seed=520552855; saved 3 raw/mask pairs
MYH10_raw_1_z-1.png | MYH10_mask_1_z-1.png | z_offset=-1
MYH10_raw_2_z0.png | MYH10_mask_2_z0.png | z_offset=0
MYH10_raw_3_z1.png | MYH10_mask_3_z1.png | z_offset=1


## NPM1


In [18]:
from pathlib import Path
import random
import sys

project_root = Path.cwd().resolve()
if not (project_root / "ProofOfConcept").exists():
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import pandas as pd
from ProofOfConcept.visualize_structures import visualize_structure_samples

structure_name = "NPM1"
metadata = pd.read_csv(project_root / "ProofOfConcept" / "metadata.csv")
random_seed = random.randint(1, 10**9)

selected = visualize_structure_samples(
    structure_name,
    metadata=metadata,
    input_dir=project_root / "ProofOfConcept" / "structure_downloads",
    output_dir=project_root / "visualizations",
    num_files=3,
    random_seed=random_seed,
    z_offsets=[-1, 0, 1],
)

print(f"{structure_name}: random_seed={random_seed}; saved {len(selected)} raw/mask pairs")
for raw_path, mask_path, z_offset in selected:
    print(f"{raw_path.name} | {mask_path.name} | z_offset={z_offset}")


C:\Users\Adithi\AppData\Local\Temp\ipykernel_19608\3643103177.py:15: DtypeWarning: Columns (0: meta_colony_centroid) have mixed types. Specify dtype option on import or set low_memory=False.
  metadata = pd.read_csv(project_root / "ProofOfConcept" / "metadata.csv")


NPM1: random_seed=58531145; saved 3 raw/mask pairs
NPM1_raw_1_z-1.png | NPM1_mask_1_z-1.png | z_offset=-1
NPM1_raw_2_z0.png | NPM1_mask_2_z0.png | z_offset=0
NPM1_raw_3_z1.png | NPM1_mask_3_z1.png | z_offset=1


## NUP153


In [19]:
from pathlib import Path
import random
import sys

project_root = Path.cwd().resolve()
if not (project_root / "ProofOfConcept").exists():
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import pandas as pd
from ProofOfConcept.visualize_structures import visualize_structure_samples

structure_name = "NUP153"
metadata = pd.read_csv(project_root / "ProofOfConcept" / "metadata.csv")
random_seed = random.randint(1, 10**9)

selected = visualize_structure_samples(
    structure_name,
    metadata=metadata,
    input_dir=project_root / "ProofOfConcept" / "structure_downloads",
    output_dir=project_root / "visualizations",
    num_files=3,
    random_seed=random_seed,
    z_offsets=[-1, 0, 1],
)

print(f"{structure_name}: random_seed={random_seed}; saved {len(selected)} raw/mask pairs")
for raw_path, mask_path, z_offset in selected:
    print(f"{raw_path.name} | {mask_path.name} | z_offset={z_offset}")


C:\Users\Adithi\AppData\Local\Temp\ipykernel_19608\120810340.py:15: DtypeWarning: Columns (0: meta_colony_centroid) have mixed types. Specify dtype option on import or set low_memory=False.
  metadata = pd.read_csv(project_root / "ProofOfConcept" / "metadata.csv")


NUP153: random_seed=588066057; saved 3 raw/mask pairs
NUP153_raw_1_z-1.png | NUP153_mask_1_z-1.png | z_offset=-1
NUP153_raw_2_z0.png | NUP153_mask_2_z0.png | z_offset=0
NUP153_raw_3_z1.png | NUP153_mask_3_z1.png | z_offset=1


## PXN


In [20]:
from pathlib import Path
import random
import sys

project_root = Path.cwd().resolve()
if not (project_root / "ProofOfConcept").exists():
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import pandas as pd
from ProofOfConcept.visualize_structures import visualize_structure_samples

structure_name = "PXN"
metadata = pd.read_csv(project_root / "ProofOfConcept" / "metadata.csv")
random_seed = random.randint(1, 10**9)

selected = visualize_structure_samples(
    structure_name,
    metadata=metadata,
    input_dir=project_root / "ProofOfConcept" / "structure_downloads",
    output_dir=project_root / "visualizations",
    num_files=3,
    random_seed=random_seed,
    z_offsets=[-1, 0, 1],
)

print(f"{structure_name}: random_seed={random_seed}; saved {len(selected)} raw/mask pairs")
for raw_path, mask_path, z_offset in selected:
    print(f"{raw_path.name} | {mask_path.name} | z_offset={z_offset}")


C:\Users\Adithi\AppData\Local\Temp\ipykernel_19608\4230583143.py:15: DtypeWarning: Columns (0: meta_colony_centroid) have mixed types. Specify dtype option on import or set low_memory=False.
  metadata = pd.read_csv(project_root / "ProofOfConcept" / "metadata.csv")


PXN: random_seed=659010077; saved 3 raw/mask pairs
PXN_raw_1_z-1.png | PXN_mask_1_z-1.png | z_offset=-1
PXN_raw_2_z0.png | PXN_mask_2_z0.png | z_offset=0
PXN_raw_3_z1.png | PXN_mask_3_z1.png | z_offset=1


## RAB5A


In [21]:
from pathlib import Path
import random
import sys

project_root = Path.cwd().resolve()
if not (project_root / "ProofOfConcept").exists():
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import pandas as pd
from ProofOfConcept.visualize_structures import visualize_structure_samples

structure_name = "RAB5A"
metadata = pd.read_csv(project_root / "ProofOfConcept" / "metadata.csv")
random_seed = random.randint(1, 10**9)

selected = visualize_structure_samples(
    structure_name,
    metadata=metadata,
    input_dir=project_root / "ProofOfConcept" / "structure_downloads",
    output_dir=project_root / "visualizations",
    num_files=3,
    random_seed=random_seed,
    z_offsets=[-1, 0, 1],
)

print(f"{structure_name}: random_seed={random_seed}; saved {len(selected)} raw/mask pairs")
for raw_path, mask_path, z_offset in selected:
    print(f"{raw_path.name} | {mask_path.name} | z_offset={z_offset}")


C:\Users\Adithi\AppData\Local\Temp\ipykernel_19608\2497046859.py:15: DtypeWarning: Columns (0: meta_colony_centroid) have mixed types. Specify dtype option on import or set low_memory=False.
  metadata = pd.read_csv(project_root / "ProofOfConcept" / "metadata.csv")


RAB5A: random_seed=568579619; saved 3 raw/mask pairs
RAB5A_raw_1_z-1.png | RAB5A_mask_1_z-1.png | z_offset=-1
RAB5A_raw_2_z0.png | RAB5A_mask_2_z0.png | z_offset=0
RAB5A_raw_3_z1.png | RAB5A_mask_3_z1.png | z_offset=1


## SEC61B


In [22]:
from pathlib import Path
import random
import sys

project_root = Path.cwd().resolve()
if not (project_root / "ProofOfConcept").exists():
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import pandas as pd
from ProofOfConcept.visualize_structures import visualize_structure_samples

structure_name = "SEC61B"
metadata = pd.read_csv(project_root / "ProofOfConcept" / "metadata.csv")
random_seed = random.randint(1, 10**9)

selected = visualize_structure_samples(
    structure_name,
    metadata=metadata,
    input_dir=project_root / "ProofOfConcept" / "structure_downloads",
    output_dir=project_root / "visualizations",
    num_files=3,
    random_seed=random_seed,
    z_offsets=[-1, 0, 1],
)

print(f"{structure_name}: random_seed={random_seed}; saved {len(selected)} raw/mask pairs")
for raw_path, mask_path, z_offset in selected:
    print(f"{raw_path.name} | {mask_path.name} | z_offset={z_offset}")


C:\Users\Adithi\AppData\Local\Temp\ipykernel_19608\4051883531.py:15: DtypeWarning: Columns (0: meta_colony_centroid) have mixed types. Specify dtype option on import or set low_memory=False.
  metadata = pd.read_csv(project_root / "ProofOfConcept" / "metadata.csv")


SEC61B: random_seed=196268549; saved 3 raw/mask pairs
SEC61B_raw_1_z-1.png | SEC61B_mask_1_z-1.png | z_offset=-1
SEC61B_raw_2_z0.png | SEC61B_mask_2_z0.png | z_offset=0
SEC61B_raw_3_z1.png | SEC61B_mask_3_z1.png | z_offset=1


## SLC25A17


In [23]:
from pathlib import Path
import random
import sys

project_root = Path.cwd().resolve()
if not (project_root / "ProofOfConcept").exists():
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import pandas as pd
from ProofOfConcept.visualize_structures import visualize_structure_samples

structure_name = "SLC25A17"
metadata = pd.read_csv(project_root / "ProofOfConcept" / "metadata.csv")
random_seed = random.randint(1, 10**9)

selected = visualize_structure_samples(
    structure_name,
    metadata=metadata,
    input_dir=project_root / "ProofOfConcept" / "structure_downloads",
    output_dir=project_root / "visualizations",
    num_files=3,
    random_seed=random_seed,
    z_offsets=[-1, 0, 1],
)

print(f"{structure_name}: random_seed={random_seed}; saved {len(selected)} raw/mask pairs")
for raw_path, mask_path, z_offset in selected:
    print(f"{raw_path.name} | {mask_path.name} | z_offset={z_offset}")


C:\Users\Adithi\AppData\Local\Temp\ipykernel_19608\2505124587.py:15: DtypeWarning: Columns (0: meta_colony_centroid) have mixed types. Specify dtype option on import or set low_memory=False.
  metadata = pd.read_csv(project_root / "ProofOfConcept" / "metadata.csv")


SLC25A17: random_seed=925902604; saved 3 raw/mask pairs
SLC25A17_raw_1_z-1.png | SLC25A17_mask_1_z-1.png | z_offset=-1
SLC25A17_raw_2_z0.png | SLC25A17_mask_2_z0.png | z_offset=0
SLC25A17_raw_3_z1.png | SLC25A17_mask_3_z1.png | z_offset=1


## SMC1A


In [24]:
from pathlib import Path
import random
import sys

project_root = Path.cwd().resolve()
if not (project_root / "ProofOfConcept").exists():
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import pandas as pd
from ProofOfConcept.visualize_structures import visualize_structure_samples

structure_name = "SMC1A"
metadata = pd.read_csv(project_root / "ProofOfConcept" / "metadata.csv")
random_seed = random.randint(1, 10**9)

selected = visualize_structure_samples(
    structure_name,
    metadata=metadata,
    input_dir=project_root / "ProofOfConcept" / "structure_downloads",
    output_dir=project_root / "visualizations",
    num_files=3,
    random_seed=random_seed,
    z_offsets=[-1, 0, 1],
)

print(f"{structure_name}: random_seed={random_seed}; saved {len(selected)} raw/mask pairs")
for raw_path, mask_path, z_offset in selected:
    print(f"{raw_path.name} | {mask_path.name} | z_offset={z_offset}")


C:\Users\Adithi\AppData\Local\Temp\ipykernel_19608\3712811110.py:15: DtypeWarning: Columns (0: meta_colony_centroid) have mixed types. Specify dtype option on import or set low_memory=False.
  metadata = pd.read_csv(project_root / "ProofOfConcept" / "metadata.csv")


SMC1A: random_seed=656790814; saved 3 raw/mask pairs
SMC1A_raw_1_z-1.png | SMC1A_mask_1_z-1.png | z_offset=-1
SMC1A_raw_2_z0.png | SMC1A_mask_2_z0.png | z_offset=0
SMC1A_raw_3_z1.png | SMC1A_mask_3_z1.png | z_offset=1


## SON


In [25]:
from pathlib import Path
import random
import sys

project_root = Path.cwd().resolve()
if not (project_root / "ProofOfConcept").exists():
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import pandas as pd
from ProofOfConcept.visualize_structures import visualize_structure_samples

structure_name = "SON"
metadata = pd.read_csv(project_root / "ProofOfConcept" / "metadata.csv")
random_seed = random.randint(1, 10**9)

selected = visualize_structure_samples(
    structure_name,
    metadata=metadata,
    input_dir=project_root / "ProofOfConcept" / "structure_downloads",
    output_dir=project_root / "visualizations",
    num_files=3,
    random_seed=random_seed,
    z_offsets=[-1, 0, 1],
)

print(f"{structure_name}: random_seed={random_seed}; saved {len(selected)} raw/mask pairs")
for raw_path, mask_path, z_offset in selected:
    print(f"{raw_path.name} | {mask_path.name} | z_offset={z_offset}")


C:\Users\Adithi\AppData\Local\Temp\ipykernel_19608\2070388900.py:15: DtypeWarning: Columns (0: meta_colony_centroid) have mixed types. Specify dtype option on import or set low_memory=False.
  metadata = pd.read_csv(project_root / "ProofOfConcept" / "metadata.csv")


SON: random_seed=706969546; saved 3 raw/mask pairs
SON_raw_1_z-1.png | SON_mask_1_z-1.png | z_offset=-1
SON_raw_2_z0.png | SON_mask_2_z0.png | z_offset=0
SON_raw_3_z1.png | SON_mask_3_z1.png | z_offset=1


## ST6GAL1


In [26]:
from pathlib import Path
import random
import sys

project_root = Path.cwd().resolve()
if not (project_root / "ProofOfConcept").exists():
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import pandas as pd
from ProofOfConcept.visualize_structures import visualize_structure_samples

structure_name = "ST6GAL1"
metadata = pd.read_csv(project_root / "ProofOfConcept" / "metadata.csv")
random_seed = random.randint(1, 10**9)

selected = visualize_structure_samples(
    structure_name,
    metadata=metadata,
    input_dir=project_root / "ProofOfConcept" / "structure_downloads",
    output_dir=project_root / "visualizations",
    num_files=3,
    random_seed=random_seed,
    z_offsets=[-1, 0, 1],
)

print(f"{structure_name}: random_seed={random_seed}; saved {len(selected)} raw/mask pairs")
for raw_path, mask_path, z_offset in selected:
    print(f"{raw_path.name} | {mask_path.name} | z_offset={z_offset}")


C:\Users\Adithi\AppData\Local\Temp\ipykernel_19608\1486134611.py:15: DtypeWarning: Columns (0: meta_colony_centroid) have mixed types. Specify dtype option on import or set low_memory=False.
  metadata = pd.read_csv(project_root / "ProofOfConcept" / "metadata.csv")


ST6GAL1: random_seed=313781406; saved 3 raw/mask pairs
ST6GAL1_raw_1_z-1.png | ST6GAL1_mask_1_z-1.png | z_offset=-1
ST6GAL1_raw_2_z0.png | ST6GAL1_mask_2_z0.png | z_offset=0
ST6GAL1_raw_3_z1.png | ST6GAL1_mask_3_z1.png | z_offset=1


## TJP1


In [27]:
from pathlib import Path
import random
import sys

project_root = Path.cwd().resolve()
if not (project_root / "ProofOfConcept").exists():
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import pandas as pd
from ProofOfConcept.visualize_structures import visualize_structure_samples

structure_name = "TJP1"
metadata = pd.read_csv(project_root / "ProofOfConcept" / "metadata.csv")
random_seed = random.randint(1, 10**9)

selected = visualize_structure_samples(
    structure_name,
    metadata=metadata,
    input_dir=project_root / "ProofOfConcept" / "structure_downloads",
    output_dir=project_root / "visualizations",
    num_files=3,
    random_seed=random_seed,
    z_offsets=[-1, 0, 1],
)

print(f"{structure_name}: random_seed={random_seed}; saved {len(selected)} raw/mask pairs")
for raw_path, mask_path, z_offset in selected:
    print(f"{raw_path.name} | {mask_path.name} | z_offset={z_offset}")


C:\Users\Adithi\AppData\Local\Temp\ipykernel_19608\2879147279.py:15: DtypeWarning: Columns (0: meta_colony_centroid) have mixed types. Specify dtype option on import or set low_memory=False.
  metadata = pd.read_csv(project_root / "ProofOfConcept" / "metadata.csv")


TJP1: random_seed=611887887; saved 3 raw/mask pairs
TJP1_raw_1_z-1.png | TJP1_mask_1_z-1.png | z_offset=-1
TJP1_raw_2_z0.png | TJP1_mask_2_z0.png | z_offset=0
TJP1_raw_3_z1.png | TJP1_mask_3_z1.png | z_offset=1


## TOMM20


In [28]:
from pathlib import Path
import random
import sys

project_root = Path.cwd().resolve()
if not (project_root / "ProofOfConcept").exists():
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import pandas as pd
from ProofOfConcept.visualize_structures import visualize_structure_samples

structure_name = "TOMM20"
metadata = pd.read_csv(project_root / "ProofOfConcept" / "metadata.csv")
random_seed = random.randint(1, 10**9)

selected = visualize_structure_samples(
    structure_name,
    metadata=metadata,
    input_dir=project_root / "ProofOfConcept" / "structure_downloads",
    output_dir=project_root / "visualizations",
    num_files=3,
    random_seed=random_seed,
    z_offsets=[-1, 0, 1],
)

print(f"{structure_name}: random_seed={random_seed}; saved {len(selected)} raw/mask pairs")
for raw_path, mask_path, z_offset in selected:
    print(f"{raw_path.name} | {mask_path.name} | z_offset={z_offset}")


C:\Users\Adithi\AppData\Local\Temp\ipykernel_19608\2964001141.py:15: DtypeWarning: Columns (0: meta_colony_centroid) have mixed types. Specify dtype option on import or set low_memory=False.
  metadata = pd.read_csv(project_root / "ProofOfConcept" / "metadata.csv")


TOMM20: random_seed=418557602; saved 3 raw/mask pairs
TOMM20_raw_1_z-1.png | TOMM20_mask_1_z-1.png | z_offset=-1
TOMM20_raw_2_z0.png | TOMM20_mask_2_z0.png | z_offset=0
TOMM20_raw_3_z1.png | TOMM20_mask_3_z1.png | z_offset=1


## TUBA1B


In [29]:
from pathlib import Path
import random
import sys

project_root = Path.cwd().resolve()
if not (project_root / "ProofOfConcept").exists():
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import pandas as pd
from ProofOfConcept.visualize_structures import visualize_structure_samples

structure_name = "TUBA1B"
metadata = pd.read_csv(project_root / "ProofOfConcept" / "metadata.csv")
random_seed = random.randint(1, 10**9)

selected = visualize_structure_samples(
    structure_name,
    metadata=metadata,
    input_dir=project_root / "ProofOfConcept" / "structure_downloads",
    output_dir=project_root / "visualizations",
    num_files=3,
    random_seed=random_seed,
    z_offsets=[-1, 0, 1],
)

print(f"{structure_name}: random_seed={random_seed}; saved {len(selected)} raw/mask pairs")
for raw_path, mask_path, z_offset in selected:
    print(f"{raw_path.name} | {mask_path.name} | z_offset={z_offset}")


C:\Users\Adithi\AppData\Local\Temp\ipykernel_19608\196908165.py:15: DtypeWarning: Columns (0: meta_colony_centroid) have mixed types. Specify dtype option on import or set low_memory=False.
  metadata = pd.read_csv(project_root / "ProofOfConcept" / "metadata.csv")


TUBA1B: random_seed=949024027; saved 3 raw/mask pairs
TUBA1B_raw_1_z-1.png | TUBA1B_mask_1_z-1.png | z_offset=-1
TUBA1B_raw_2_z0.png | TUBA1B_mask_2_z0.png | z_offset=0
TUBA1B_raw_3_z1.png | TUBA1B_mask_3_z1.png | z_offset=1
